# Project 1 — Basis Function Approximation
- Target: $f(t) = (t-\pi/2)^2$.
- Basis is: $\{1,\; t,\;t^2,\; t^3,\; t^4\}$.


In [ ]:
# ============================================================================
# PROJECT 1 - Basis Function Approximation
# Zane Mullinax
# Target: (t - 2/pi)^2
# Basis:  {1, t, t^2, t^3, t^4}
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
def trapezoidal_integration(t, y):
    # Prefer NumPy 2.x function name if present, else fallback
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, t)
    return np.trapz(y, t)


rng = np.random.default_rng(42)


# ============================================================================
t_demo = np.arange(0, np.pi, 0.1)
n_demo = len(t_demo)
true_demo = np.cos(t_demo) * t_demo
demo_std = np.std(true_demo)
noise_scale_demo = demo_std / max(rng.random(), 1e-8)       # like professor’s
x_demo = true_demo + noise_scale_demo * rng.random(n_demo)

plt.figure(figsize=(8, 6))
plt.plot(t_demo, x_demo, color='gray', label='Noisy Time Series', marker='o', linestyle='-', markersize=4)
plt.title('Synthetic Time Series with Trend and Noise', fontsize=14)
plt.xlabel('Time (t)', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()


# ============================================================================

print("=" * 70)
print("COMPLETE BASIS FUNCTION APPROXIMATION EXAMPLE")
print("=" * 70)
print()

# ----------------------------------------------------------------------------
# STEP 1: Setting up the problem
# ----------------------------------------------------------------------------
print("STEP 1: Setting up the problem")
print("-" * 70)

t = np.linspace(0, np.pi, 200)   # dense sampling on [0, π]
n_points = len(t)

# Your target (CLEAN) signal
true_signal = (t - 2.0/np.pi)**2

# noise added to create a "measured" target
signal_std = np.std(true_signal)
noise_level = 0.15                          # tweak: 0.10–0.20 works well
noise = noise_level * signal_std * rng.standard_normal(n_points)  # zero-mean
target = true_signal + noise

print(f"Domain: t ∈ [0, π] with {n_points} points")
print("Target function: f(t) = (t - 2/π)^2  with zero-mean noise added")
print(f"(noise_level ≈ {noise_level:.2f})")
print()

# ----------------------------------------------------------------------------
# STEP 2: Defining basis functions
# ----------------------------------------------------------------------------
print("STEP 2: Defining basis functions")
print("-" * 70)
n_basis = 5
basis_matrix = np.zeros((n_points, n_basis))
basis_matrix[:, 0] = 1.0      # u1(t) = 1
basis_matrix[:, 1] = t        # u2(t) = t
basis_matrix[:, 2] = t**2     # u3(t) = t^2
basis_matrix[:, 3] = t**3     # u4(t) = t^3
basis_matrix[:, 4] = t**4     # u5(t) = t^4

print(f"Number of basis functions: {n_basis}")
print("Basis functions: u₁=1, u₂=t, u₃=t², u₄=t³, u₅=t⁴")
print(f"Basis matrix shape: {basis_matrix.shape} (rows=time points, cols=basis)")
print()

# ----------------------------------------------------------------------------
# STEP 3: Computing coefficient matrix A (inner products ⟨u_i, u_j⟩)
# ----------------------------------------------------------------------------
print("STEP 3: Computing coefficient matrix A")
print("-" * 70)
A = np.zeros((n_basis, n_basis))
for i in range(n_basis):
    u_i = basis_matrix[:, i]
    for j in range(n_basis):
        u_j = basis_matrix[:, j]
        A[i, j] = trapezoidal_integration(t, u_i * u_j)
A = 0.5 * (A + A.T)
print("Matrix A (inner products of basis functions):")
print(A)
print()

# ----------------------------------------------------------------------------
# STEP 4: Computing right-hand side vector b (⟨u_i, target⟩)
# ----------------------------------------------------------------------------
print("STEP 4: Computing right-hand side vector b")
print("-" * 70)
b = np.zeros(n_basis)
for i in range(n_basis):
    b[i] = trapezoidal_integration(t, basis_matrix[:, i] * target)
    print(f"  b[{i}] = {b[i]:.6f}")
print()
print("Vector b:")
print(b)
print()

# ----------------------------------------------------------------------------
# STEP 5: Solving for coefficients (Ac = b)
# ----------------------------------------------------------------------------
print("STEP 5: Solving for coefficients (A*c = b)")
print("-" * 70)
c = np.linalg.solve(A, b)
for i in range(n_basis):
    print(f"  c_{i+1} = {c[i]:.8f}")
print()

# ----------------------------------------------------------------------------
# STEP 6: Building the approximation
# ----------------------------------------------------------------------------
print("STEP 6: Building the approximation")
print("-" * 70)
f_approx = basis_matrix @ c
print("Approximation: f_approx = Σ cᵢ · uᵢ(t)")
print(f"f_approx(t) = {c[0]:.6f} + {c[1]:.6f}t + {c[2]:.6f}t² + {c[3]:.6f}t³ + {c[4]:.6f}t⁴")
print()

# ----------------------------------------------------------------------------
# STEP 7: Checking error
# ----------------------------------------------------------------------------
print("STEP 7: Checking error")
print("-" * 70)
error = true_signal - f_approx
max_error  = np.max(np.abs(error))
mean_error = np.mean(np.abs(error))
rms_error  = np.sqrt(np.mean(error**2))
print(f"Max error:  {max_error:.6f}")
print(f"Mean error: {mean_error:.6f}")
print(f"RMS error:  {rms_error:.6f}")
print()

# ----------------------------------------------------------------------------
# STEP 8: Plotting results
# ----------------------------------------------------------------------------
print("STEP 8: Plotting results")
print("-" * 70)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: original clean target vs approximation
ax1.plot(t, true_signal, '-', lw=2, label='Original: (t - 2/π)²', alpha=0.85)
ax1.plot(t, f_approx,  '--', lw=2.2, label='Approximation (5 basis)')
ax1.set_xlabel('t')
ax1.set_ylabel('Value')
ax1.set_title('Function Approximation')
ax1.legend()
ax1.grid(alpha=0.3)

# Right: error (relative to the noisy target actually used for fitting)
ax2.plot(t, error, '-', lw=2)
ax2.axhline(0, color='gray', linestyle='--', lw=1)
ax2.fill_between(t, error, alpha=0.25)
ax2.set_xlabel('t')
ax2.set_ylabel('Error')
ax2.set_title(f'Error vs. Noisy Target (max: {max_error:.4f})')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

